In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics.classification import Accuracy
import lightning as L

# --- Load paths and labels from text files ---
root = "../fiot_highway2-main"
def read_split(fname):
    paths, labels = [], []
    with open(os.path.join(root, fname)) as f:
        for line in f:
            rel, label = line.strip().split()
            paths.append(os.path.join(root, rel))
            labels.append(int(label))
    return paths, labels

train_paths, train_labels = read_split("train.txt")
test_paths,  test_labels  = read_split("test.txt")

# --- Load data into memory (normalizing each sample) ---
def load_arrays(paths, labels):
    arrays, lbls = [], []
    for p, y in zip(paths, labels):
        a = np.load(p)
        a = (a - a.min()) / (a.max() - a.min() + 1e-8)
        arrays.append(a)
        lbls.append(y)
    X = torch.tensor(np.stack(arrays), dtype=torch.float32).unsqueeze(1)  # (B,1,H,W)
    y = torch.tensor(lbls, dtype=torch.long)
    return TensorDataset(X, y)

train_ds = load_arrays(train_paths, train_labels)
test_ds  = load_arrays(test_paths,  test_labels)
val_ds   = test_ds  # reuse test set for validation

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
test_loader  = DataLoader(test_ds,  batch_size=32)

# --- Simple CNN Model ---
class RFICNN(L.LightningModule):
    def __init__(self, num_classes=9, lr=1e-3, wd=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,5,padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc = nn.Linear(64, num_classes)
        self.loss_fn = nn.CrossEntropyLoss()
        self.acc = Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x): 
        x = torch.flatten(self.net(x), 1)
        return self.fc(x)

    def _shared(self, batch, stage):
        x,y = batch
        logits = self(x)
        loss = self.loss_fn(logits,y)
        preds = logits.argmax(1)
        acc = self.acc(preds,y)
        self.log(f"{stage}_loss", loss, prog_bar=(stage!="train"))
        self.log(f"{stage}_acc", acc,  prog_bar=True)
        return loss

    def training_step(self,batch,idx): return self._shared(batch,"train")
    def validation_step(self,batch,idx): self._shared(batch,"val")
    def test_step(self,batch,idx): self._shared(batch,"test")

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.wd)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "monitor": "val_loss"}}

# --- Train + Test ---
if __name__ == "__main__":
    model = RFICNN()
    trainer = L.Trainer(max_epochs=15, accelerator="auto", devices="auto", log_every_n_steps=10)
    trainer.fit(model, train_loader, val_loader)
    trainer.test(model, dataloaders=test_loader)

/usr/local/lib/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
2025-10-28 13:19:56.550008: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761671996.805279    2204 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761671996.877484    2204 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has a

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
